# 탱크별 온도 vs 제어요소 (heat / cool / exchanger / feeding) — 로컬 CSV 기반

InfluxDB에 매번 용로하지 않고, `Tank_Temp_Factor_Analysis.py` 로 이미 받아둔 `extracted_csv/*_Temp_vs_Factors_*.csv` 를 읽어서 pandas 로 시간 범위만 바꿔가며 그래프를 보는 노트북입니다.

**사용법**
1. 위에서 아래로 순서대로 한 번 실행 (함수/설정 셀들)
2. 각 탱크 섹션에서 `START`/`END` 문자열만 바꿔가며 해당 셀을 다시 실행하면 됩니다 (`None` = 끝까지/처음부터)
3. 새로 받은 CSV 로 갱신하고 싶으면 `load_tank_df(tank, refresh=True)` 를 다시 불러오면 됩니다

⚠️ 이 노트북은 **parkkt 폴더에서 실행**된다고 가정합니다 (working directory 기준으로 `extracted_csv/`, `analysis_out/` 을 찾습니다).

## 0. 설정 및 공통 함수

In [ ]:
%matplotlib inline
import glob
import os

import matplotlib
import matplotlib.pyplot as plt
import pandas as pd

# 한글 라벨이 깨지지 않게 (Windows 기본 탑재 폰트).
matplotlib.rcParams["font.family"] = "Malgun Gothic"
matplotlib.rcParams["axes.unicode_minus"] = False

SAVE_DIR = "extracted_csv"
OUT_DIR = "analysis_out"

In [ ]:
TANKS = {
    "P1": {
        "temp": "Scale_Out___TT_P1",
        "heat": "워킹 Tank 1 BACK Heater",
        "cool": "워킹 Tank 1 BACK Cooling SOL",
        "exchanger": "워킹 Tank 1 BACK Exchanger SOL",
        "feeding": "워킹 Tank 1 BACK Feeding SOL",
    },
    "P2": {
        "temp": "Scale_Out___TT_P2",
        "heat": "워킹 Tank 2-1 SOFT  Heater",
        "cool": "워킹 Tank 2-1 SOFT Cooling SOL",
        "exchanger": "워킹 Tank 2-1 SOFT Exchanger SOL",
        "feeding": "워킹 Tank 2-1 SOFT  Feeding SOL",
    },
    "P3": {
        "temp": "Scale_Out___TT_P3",
        "heat": "워킹 Tank 3-1 고탄성  Heater",
        "cool": "워킹 Tank 3-1 고탄성 Cooling SOL",
        "exchanger": "워킹 Tank 3-1 고탄성 Exchanger SOL",
        "feeding": "워킹 Tank 3-1 고탄성  Feeding SOL",
    },
    "P4": {
        "temp": "Scale_Out___TT_P4",
        "heat": "워킹 Tank 2-2 HARD  Heater",
        "cool": "워킹 Tank 2-2 HARD Cooling SOL",
        "exchanger": "워킹 Tank 2-2 HARD Exchanger SOL",
        "feeding": "워킹 Tank 2-2 HARD  Feeding SOL",
    },
    "P5": {
        "temp": "Scale_Out___TT_P5",
        "heat": "워킹 Tank 3-2 CUSH  Heater",
        "cool": "워킹 Tank 3-2 CUSH Cooling SOL",
        "exchanger": "워킹 Tank 3-2 CUSH Exchanger SOL",
        "feeding": "워킹 Tank 3-2 CUSH  Feeding SOL",
    },
    "I1": {
        "temp": "Scale_Out___TT_I1",
        "heat": "워킹 Tank 1 ISO Heater",
        "cool": "워킹 Tank 1 ISO Cooling SOL",
        "exchanger": "워킹 Tank 1 ISO Exchanger SOL",
        "feeding": "워킹 Tank 1 ISO Feeding SOL",
    },
    "I2": {
        "temp": "Scale_Out___TT_I2",
        "heat": "워킹 Tank 2 MDI  Heater",
        "cool": "워킹 Tank 2-2 MDI Cooling SOL",
        "exchanger": "워킹 Tank 2-2 MDI Exchanger SOL",
        "feeding": "워킹 Tank 2  MDI Feeding SOL",
    },
    "I3": {
        "temp": "Scale_Out___TT_I3",
        "heat": "워킹 Tank 3  ISO  Heater",
        "cool": "워킹 Tank 3  ISO Cooling SOL",
        "exchanger": "워킹 Tank 3  ISO Exchanger SOL",
        "feeding": "워킹 Tank 3  ISO  Feeding SOL",
    },
}

In [ ]:
FACTOR_NAMES = ("heat", "cool", "exchanger", "feeding")
COLORS = {"heat": "#d62728", "cool": "#1f77b4", "exchanger": "#2ca02c", "feeding": "#9467bd"}

In [ ]:
def find_latest_csv(tank_key):
    pattern = os.path.join(SAVE_DIR, f"{tank_key}_Temp_vs_Factors_*.csv")
    files = sorted(glob.glob(pattern))
    if not files:
        raise FileNotFoundError(
            f"{pattern} 에 맞는 CSV가 없습니다. "
            "먼저 Tank_Temp_Factor_Analysis.py 를 돌려서 데이터를 받아두세요."
        )
    return files[-1]


_CACHE = {}


def load_tank_df(tank_key, refresh=False):
    """CSV를 한 번만 읽어서 메모리에 캐싱한다. 새로 받은 CSV를 쓰려면 refresh=True."""
    if refresh or tank_key not in _CACHE:
        path = find_latest_csv(tank_key)
        df = pd.read_csv(path, index_col="Time", parse_dates=True)
        _CACHE[tank_key] = df
        print(f"[{tank_key}] {os.path.basename(path)} 로드 — {df.index.min()} ~ {df.index.max()} ({len(df)}행)")
    return _CACHE[tank_key]

In [ ]:
def plot_tank_range(tank_key, start=None, end=None, save=False):
    """start/end 는 '2026-09-10 06:00:00' 같은 문자열이나 None(처음/끝까지).
    df.loc[start:end] 로 그냥 슬라이싱하는 거라 범위 바꿔서 계속 다시 그려보면 된다.
    """
    cfg = TANKS[tank_key]
    df_full = load_tank_df(tank_key)
    df = df_full.loc[start:end]

    if df.empty:
        print(f"⚠️ [{tank_key}] 이 범위엔 데이터가 없습니다: {start} ~ {end}")
        return

    n_factors = len(FACTOR_NAMES)
    fig, axes = plt.subplots(
        1 + n_factors, 1, sharex=True, figsize=(16, 3 + 1.3 * n_factors),
        gridspec_kw={"height_ratios": [3] + [1] * n_factors},
    )

    temp_tag = cfg["temp"]
    ax_temp = axes[0]
    ax_temp.plot(df.index, df[temp_tag], color="#d62728", linewidth=0.9)
    ax_temp.set_ylabel(temp_tag, fontsize=9)
    ax_temp.set_title(f"{tank_key}   {df.index.min()} ~ {df.index.max()}   ({len(df)}행)")
    ax_temp.grid(axis="y", alpha=0.3)

    heat = df[cfg["heat"]]
    cool = df[cfg["cool"]]
    both_off = (heat == 0) & (cool == 0)
    both_on = (heat == 1) & (cool == 1)
    ylim = ax_temp.get_ylim()
    if both_off.any():
        ax_temp.fill_between(df.index, ylim[0], ylim[1], where=both_off.values,
                              color="orange", alpha=0.15, label="heat/cool 둘 다 OFF", step="post")
    if both_on.any():
        ax_temp.fill_between(df.index, ylim[0], ylim[1], where=both_on.values,
                              color="purple", alpha=0.15, label="heat/cool 둘 다 ON", step="post")
    if both_off.any() or both_on.any():
        ax_temp.legend(loc="upper right", fontsize=8)

    for ax, name in zip(axes[1:], FACTOR_NAMES):
        tag = cfg[name]
        ax.step(df.index, df[tag], where="post", color=COLORS[name], linewidth=1.0)
        ax.set_ylim(-0.2, 1.2)
        ax.set_yticks([0, 1])
        ax.set_ylabel(name, fontsize=9)
        ax.set_title(tag, loc="right", fontsize=8, color="#666666", pad=2)
        ax.grid(axis="y", alpha=0.3)

    fig.autofmt_xdate()
    fig.tight_layout()

    if save:
        os.makedirs(OUT_DIR, exist_ok=True)
        out_path = os.path.join(OUT_DIR, f"{tank_key}_Temp_vs_Factors_range.png")
        fig.savefig(out_path, dpi=130)
        print(f"🖼  저장: {out_path}")

    print(f"heat/cool 둘다OFF={int(both_off.sum())}  둘다ON={int(both_on.sum())}")

## 1. 전체 탱크 미리 로드 (보유 중인 데이터 범위 확인)

In [ ]:
DATA = {tank: load_tank_df(tank) for tank in TANKS}

## 2. P1 — 워킹 Tank 1 BACK

In [ ]:
P1_START = None  # 예: "2026-09-10 06:00:00"
P1_END = None    # 예: "2026-09-10 18:00:00"

plot_tank_range("P1", start=P1_START, end=P1_END)

## 3. P2 — 워킹 Tank 2-1 SOFT

In [ ]:
P2_START = None  # 예: "2026-09-10 06:00:00"
P2_END = None    # 예: "2026-09-10 18:00:00"

plot_tank_range("P2", start=P2_START, end=P2_END)

## 4. P3 — 워킹 Tank 3-1 고탄성

In [ ]:
P3_START = None  # 예: "2026-09-10 06:00:00"
P3_END = None    # 예: "2026-09-10 18:00:00"

plot_tank_range("P3", start=P3_START, end=P3_END)

## 5. P4 — 워킹 Tank 2-2 HARD

In [ ]:
P4_START = None  # 예: "2026-09-10 06:00:00"
P4_END = None    # 예: "2026-09-10 18:00:00"

plot_tank_range("P4", start=P4_START, end=P4_END)

## 6. P5 — 워킹 Tank 3-2 CUSH

In [ ]:
P5_START = None  # 예: "2026-09-10 06:00:00"
P5_END = None    # 예: "2026-09-10 18:00:00"

plot_tank_range("P5", start=P5_START, end=P5_END)

## 7. I1 — 워킹 Tank 1 ISO

In [ ]:
I1_START = None  # 예: "2026-09-10 06:00:00"
I1_END = None    # 예: "2026-09-10 18:00:00"

plot_tank_range("I1", start=I1_START, end=I1_END)

## 8. I2 — 워킹 Tank 2 MDI

In [ ]:
I2_START = None  # 예: "2026-09-10 06:00:00"
I2_END = None    # 예: "2026-09-10 18:00:00"

plot_tank_range("I2", start=I2_START, end=I2_END)

## 9. I3 — 워킹 Tank 3 ISO

In [ ]:
I3_START = None  # 예: "2026-09-10 06:00:00"
I3_END = None    # 예: "2026-09-10 18:00:00"

plot_tank_range("I3", start=I3_START, end=I3_END)